# Customer Churn Prediction — Evaluation

**Goal:** Load the best model from training, evaluate on the held-out test set, and produce:
- Classification metrics (ROC-AUC, PR-AUC, F1, Precision, Recall)
- Confusion matrix, ROC curve, Precision-Recall curve
- SHAP explanations (summary, bar, waterfall plots)
- Feature importance ranking

> **Prerequisite:** Run `notebooks/02-train.ipynb` first to generate `models/best_model.joblib`

## 0. Setup & Imports

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Project root detection ────────────────────────────────────────────────
def _find_project_root():
    here = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(here, 'data', 'raw', 'cell2cell.csv')):
            return here
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    return os.getcwd()

ROOT = Path(_find_project_root())
sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

# ── Paths ──────────────────────────────────────────────────────────────────
MODELS_DIR = ROOT / 'models'
FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print(f"Models dir: {MODELS_DIR}")
print(f"Figures dir: {FIGURES_DIR}")

# Consistent plot style
sns.set_style('whitegrid')
CB_COLORS = ['#0072B2', '#D55E00', '#009E73', '#F0E442',
             '#CC79A7', '#56B4E9', '#E69F00', '#000000']

## 1. Load Test Data & Trained Artifacts

In [ ]:
from src.data.load import load_data, split_data
from src.config import Paths as ConfigPaths

# Load and split data
df = load_data()
_, X_test, _, y_test = split_data(df)
print(f"Test set: {len(X_test):,} samples")
print(f"Test churn rate: {y_test.mean():.3f}")

In [ ]:
import joblib

# Load preprocessor
preproc_path = MODELS_DIR / 'preprocessor.joblib'
if not preproc_path.exists():
    raise FileNotFoundError(f"Preprocessor not found at {preproc_path}. "
                            f"Run 02-train.ipynb first.")
preprocessor = joblib.load(preproc_path)
print(f"✓ Preprocessor loaded from: {preproc_path}")

# Load best model
model_path = MODELS_DIR / 'best_model.joblib'
if not model_path.exists():
    raise FileNotFoundError(f"Model not found at {model_path}. "
                            f"Run 02-train.ipynb first.")
model = joblib.load(model_path)
print(f"✓ Model loaded from: {model_path}")
print(f"  Model type: {type(model).__name__}")

# Load metadata (if available)
meta_path = MODELS_DIR / 'best_model_metadata.json'
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    print(f"  Best algorithm: {meta.get('algorithm', 'N/A')}")
    print(f"  Strategy: {meta.get('strategy', 'N/A')}")
    print(f"  Training PR-AUC: {meta.get('pr_auc', 'N/A')}")

# Load feature names
feat_path = MODELS_DIR / 'feature_names.txt'
if feat_path.exists():
    with open(feat_path) as f:
        feature_names = [line.strip() for line in f if line.strip()]
    print(f"  Features: {len(feature_names)}")
else:
    feature_names = None
    print("  (No feature_names.txt found)")

## 2. Preprocess Test Data

In [ ]:
X_test_processed = preprocessor.transform(X_test)

# Convert to DataFrame for SHAP compatibility
if hasattr(X_test_processed, 'toarray'):
    X_test_processed = X_test_processed.toarray()

if feature_names is not None:
    X_test_df = pd.DataFrame(
        X_test_processed,
        columns=feature_names,
        index=X_test.index,
    )
else:
    X_test_df = pd.DataFrame(X_test_processed)

print(f"Test set after preprocessing: {X_test_df.shape}")
print(f"NaN values: {X_test_df.isna().sum().sum()}")

## 3. Generate Predictions

In [ ]:
y_pred = model.predict(X_test_df)

y_proba = model.predict_proba(X_test_df)[:, 1]

print(f"Predictions shape: {y_pred.shape}")
print(f"Predicted churn rate: {y_pred.mean():.3f}  (actual: {y_test.mean():.3f})")
print(f"\nPrediction distribution:\n  Stay (0): {(y_pred == 0).sum():,}")
print(f"  Churn (1): {(y_pred == 1).sum():,}")

## 4. Classification Metrics

| Metric | Description |
|---|---|
| **ROC-AUC** | Area under the ROC curve — overall rank quality |
| **PR-AUC** | Area under Precision-Recall curve — best for imbalanced data |
| **F1** | Harmonic mean of precision & recall |
| **Precision** | TP / (TP + FP) — how many predicted churns were correct |
| **Recall** | TP / (TP + FN) — how many actual churns were caught |

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve,
)

metrics = {
    'ROC-AUC': float(roc_auc_score(y_test, y_proba)),
    'PR-AUC':  float(average_precision_score(y_test, y_proba)),
    'F1':      float(f1_score(y_test, y_pred)),
    'Precision': float(precision_score(y_test, y_pred)),
    'Recall':   float(recall_score(y_test, y_pred)),
}

print("=" * 50)
print("  TEST SET METRICS")
print("=" * 50)
for k, v in metrics.items():
    print(f"  {k:>12s} : {v:.4f}")
print("=" * 50)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Stay', 'Churn']))

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', cbar=False,
    xticklabels=['Predicted\nStay', 'Predicted\nChurn'],
    yticklabels=['Actual\nStay', 'Actual\nChurn'],
    ax=ax,
)
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()

cm_path = FIGURES_DIR / 'confusion_matrix.png'
fig.savefig(cm_path, dpi=150)
plt.show()
print(f"Saved to: {cm_path}")

# Decompose
tn, fp, fn, tp = cm.ravel()
print(f"\n  True Negatives : {tn:>6,}")
print(f"  False Positives: {fp:>6,}")
print(f"  False Negatives: {fn:>6,}")
print(f"  True Positives : {tp:>6,}")

## 6. ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color=CB_COLORS[0], lw=2,
        label=f"ROC (AUC = {metrics['ROC-AUC']:.4f})")
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', alpha=0.7)
ax.fill_between(fpr, tpr, alpha=0.1, color=CB_COLORS[0])
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()

roc_path = FIGURES_DIR / 'roc_curve.png'
fig.savefig(roc_path, dpi=150)
plt.show()
print(f"Saved to: {roc_path}")

## 7. Precision-Recall Curve

PR curves are more informative than ROC for imbalanced datasets.

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_proba)

# Baseline: proportion of positives
baseline = y_test.mean()

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(recall, precision, color=CB_COLORS[1], lw=2,
        label=f"PR (AUC = {metrics['PR-AUC']:.4f})")
ax.axhline(y=baseline, color='gray', lw=1, linestyle='--', alpha=0.7,
           label=f'Baseline ({baseline:.3f})')
ax.fill_between(recall, precision, alpha=0.1, color=CB_COLORS[1])
ax.set_xlabel('Recall', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curve', fontsize=13, fontweight='bold')
ax.legend(loc='lower left', fontsize=10)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
plt.tight_layout()

pr_path = FIGURES_DIR / 'pr_curve.png'
fig.savefig(pr_path, dpi=150)
plt.show()
print(f"Saved to: {pr_path}")

## 8. SHAP Explanations

We use SHAP (SHapley Additive exPlanations) to understand which features drive predictions.
Three plots are generated:
1. **Summary plot** — beeswarm of SHAP values per feature (blue = low value, red = high value)
2. **Bar plot** — mean absolute SHAP values (global feature importance)
3. **Waterfall plot** — decomposition of a single prediction

In [ ]:
try:
    import shap
    
    # Determine if tree-based (faster TreeExplainer) or not
    model_type = type(model).__name__.lower()
    is_tree = any(hint in model_type for hint in ['randomforest', 'xgb', 'xgboost', 
                                                   'lightgbm', 'catboost', 'gradientboosting'])
    
    # Unwrap from imblearn pipeline if needed
    if hasattr(model, 'named_steps') and 'classifier' in model.named_steps:
        inner_model = model.named_steps['classifier']
        inner_type = type(inner_model).__name__.lower()
        is_tree = is_tree or any(hint in inner_type for hint in 
                                 ['randomforest', 'xgb', 'xgboost', 'lightgbm', 'catboost', 'gradientboosting'])
    else:
        inner_model = model
    
    if is_tree:
        print("Using TreeExplainer (fast path)...")
        explainer = shap.TreeExplainer(inner_model)
    else:
        print("Using KernelExplainer (may take a moment)...")
        # Use a subset for KernelExplainer background
        background = X_test_df.sample(n=min(100, len(X_test_df)), random_state=42)
        explainer = shap.KernelExplainer(inner_model.predict_proba, background)
    
    # Compute SHAP values (use subset for speed if test is large)
    n_explain = min(500, len(X_test_df))
    X_explain = X_test_df.sample(n=n_explain, random_state=42) if n_explain < len(X_test_df) else X_test_df
    print(f"Computing SHAP values on {len(X_explain)} samples...")
    
    shap_values = explainer.shap_values(X_explain)
    
    # Handle multi-class output
    if isinstance(shap_values, list):
        shap_values = np.array(shap_values[1])  # class 1 = churn
    elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]
    
    print(f"SHAP values shape: {shap_values.shape}")
    
    # ── Summary plot (beeswarm) ───────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X_explain, feature_names=X_explain.columns,
                      show=False, max_display=15, ax=ax)
    plt.tight_layout()
    summary_path = FIGURES_DIR / 'shap_summary.png'
    fig.savefig(summary_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Summary plot saved to: {summary_path}")
    
    # ── Bar plot (global importance) ───────────────────────────────────────
    mean_abs = np.abs(shap_values).mean(axis=0)
    order = np.argsort(mean_abs)[::-1]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.bar_plot(
        mean_abs[order][:15],
        feature_names=np.array(X_explain.columns)[order][:15],
        show=False,
        ax=ax,
    )
    ax.set_title('Top 15 Features by Mean |SHAP Value|', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    bar_path = FIGURES_DIR / 'shap_bar.png'
    fig.savefig(bar_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Bar plot saved to: {bar_path}")
    
    # ── Waterfall plot (single prediction) ─────────────────────────────────
    exp = shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value[1],
        data=X_explain.iloc[0].values,
        feature_names=X_explain.columns.tolist(),
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    shap.plots.waterfall(exp, show=False, max_display=12)
    plt.tight_layout()
    waterfall_path = FIGURES_DIR / 'shap_waterfall.png'
    fig.savefig(waterfall_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Waterfall plot saved to: {waterfall_path}")
    
    shap_success = True
    
except ImportError:
    print("SHAP not installed. Install with: uv pip install shap")
    shap_success = False
except Exception as e:
    print(f"SHAP analysis failed: {e}")
    shap_success = False

## 9. Feature Importance (Model-Specific)

For tree-based models, extract built-in `feature_importances_`. For linear models, use coefficient magnitude.

In [ ]:
# Get the underlying estimator (unwrap SMOTE pipeline)
if hasattr(model, 'named_steps') and 'classifier' in model.named_steps:
    inner = model.named_steps['classifier']
else:
    inner = model

importance_series = None

# Tree-based models
if hasattr(inner, 'feature_importances_'):
    importance_series = pd.Series(
        inner.feature_importances_,
        index=X_test_df.columns,
    ).sort_values(ascending=False)
    title = 'Feature Importance (feature_importances_)'

# Linear models (logistic regression)
elif hasattr(inner, 'coef_'):
    coef = inner.coef_.flatten()
    importance_series = pd.Series(
        np.abs(coef),
        index=X_test_df.columns,
    ).sort_values(ascending=False)
    title = 'Feature Importance (|coefficient|)'

if importance_series is not None:
    # Top 20 features
    top20 = importance_series.head(20)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(range(len(top20)), top20.values, color=CB_COLORS[4])
    ax.set_yticks(range(len(top20)))
    ax.set_yticklabels(top20.index)
    ax.invert_yaxis()
    ax.set_xlabel('Importance', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for bar, val in zip(bars, top20.values):
        ax.text(val + val * 0.02, bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=8)
    
    plt.tight_layout()
    feat_path = FIGURES_DIR / 'feature_importance.png'
    fig.savefig(feat_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Feature importance plot saved to: {feat_path}")
    
    # Print table
    print(f"\n{'='*60}")
    print(f"  TOP 20 FEATURES")
    print(f"{'='*60}")
    for i, (feat, val) in enumerate(top20.items(), 1):
        print(f"  {i:>2d}. {feat:<45s} {val:.4f}")
    print(f"{'='*60}")
else:
    print("No built-in feature importance available for this model type.")
    print(f"Model type: {type(inner).__name__}")

## 10. Save Evaluation Results

Persist metrics and prediction details for reference.

In [ ]:
# Save metrics as JSON
metrics_path = MODELS_DIR / 'evaluation_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"✓ Metrics saved to: {metrics_path}")

# Save predictions alongside actual values
pred_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_pred,
    'probability': y_proba,
}, index=y_test.index)
pred_path = MODELS_DIR / 'test_predictions.csv'
pred_df.to_csv(pred_path, index=False)
print(f"✓ Predictions saved to: {pred_path}")

print(f"\n{'='*60}")
print(f"  Evaluation complete.")
print(f"  All figures saved to: {FIGURES_DIR}")
print(f"{'='*60}")

## 11. Summary

| Artifact | Path |
|---|---|
| Test metrics | `models/evaluation_metrics.json` |
| Confusion matrix | `reports/figures/confusion_matrix.png` |
| ROC curve | `reports/figures/roc_curve.png` |
| PR curve | `reports/figures/pr_curve.png` |
| SHAP summary | `reports/figures/shap_summary.png` |
| SHAP bar plot | `reports/figures/shap_bar.png` |
| SHAP waterfall | `reports/figures/shap_waterfall.png` |
| Feature importance | `reports/figures/feature_importance.png` |
| Test predictions | `models/test_predictions.csv` |

### Key Takeaways
- **PR-AUC** is the primary metric (imbalanced dataset with ~29% churn)
- **Recall** tells us what fraction of churners we catch
- **Precision** tells us how many of our churn alerts are correct
- SHAP plots reveal which features drive churn predictions